In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
import pickle
from src import (
    set_plot_style
)

colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

## 2. Configuration

Set `DATA_TYPE` and `PICKING_METHOD` to analyze specific signal type and phase picker combination.

**Available combinations:**
- Signal types: `acceleration`, `velocity`, `displacement`
- Pickers: `ar_pick`, `phasenet`

**Total analyses:** 3 signal types × 2 pickers × 4 coda methods = **24 result sets**

This notebook processes one combination at a time. Results are organized in separate directories to enable systematic comparison.

In [ ]:
# CONFIGURATION
#EVENT_ID = 'INT-41004391'
EVENT_ID = 'IT-2009-0009'
DATA_TYPE = 'displacement'  # Options: 'acceleration', 'velocity', 'displacement'

# Determine signal column name and units based on DATA_TYPE
if DATA_TYPE == 'acceleration':
    SIGNAL_COLUMN = 'signal'
    SIGNAL_UNIT = 'cm/s²'
    PEAK_COLUMN = 'PGA_CM/S^2'
    TIME_PEAK_COLUMN = 'TIME_PGA_S'
elif DATA_TYPE == 'velocity':
    SIGNAL_COLUMN = 'signal'
    SIGNAL_UNIT = 'cm/s'
    PEAK_COLUMN = 'PGV_CM/S'
    TIME_PEAK_COLUMN = 'TIME_PGV_S'
elif DATA_TYPE == 'displacement':
    SIGNAL_COLUMN = 'signal'
    SIGNAL_UNIT = 'cm'
    PEAK_COLUMN = 'PGD_CM'
    TIME_PEAK_COLUMN = 'TIME_PGD_S'
else:
    raise ValueError(f"Unknown DATA_TYPE: {DATA_TYPE}")

logger.info(f"Working with {DATA_TYPE} data")
logger.info(f"Signal column: {SIGNAL_COLUMN}")
logger.info(f"Peak column: {PEAK_COLUMN}")

## 3. Data Loading

Load windowed signals for **all four coda detection methods** (Rautian, Arias, Envelope, Median).

### Input File Structure

Each pickle file contains a nested dictionary:

```python
windowed_signals = {
    'STATION_CODE': {
        'HNE': {
            'pre_event': {
                'signal': array([...]),      # Signal values
                'time': array([...]),        # Time axis (seconds)
                'start_samples': int,        # Window start (samples)
                'end_samples': int,          # Window end (samples)
                'start_seconds': float,      # Window start (seconds)
                'end_seconds': float,        # Window end (seconds)
                'duration_samples': int,     # Duration (samples)
                'duration_seconds': float    # Duration (seconds)
            },
            'p_wave': {...},
            's_wave': {...},
            'coda': {...}
        },
        'HNN': {...},
        'HNZ': {...}
    },
    ...
}
```

**Key features:**
- **4 windows per component** (pre-event, P, S, coda)
- **Dual representation:** Both samples (for computation) and seconds (for output)
- **Adaptive window starts:** Different $t_0$ per station/component
